In [1]:
import torch

# Fix for older PyTorch versions lacking the float8 attribute required by newer bitsandbytes
if not hasattr(torch, "float8_e8m0fnu"):
    setattr(torch, "float8_e8m0fnu", torch.float32)

# Stage 00b2 — Descriptions CSV + Figure Cropping & Label Matching

**What this notebook does:**

1. Exports all *Brief Description of the Drawings* texts from the PatSeer Excel
   into a single `data/descriptions.csv`.

2. Detects and crops every sub-figure on each drawing sheet using
   **DocLayout-YOLO** (document layout detector) + **EasyOCR** (label reader) —
   free, local, GPU-accelerated.

The matching logic:
1. DocLayout-YOLO detects `figure` and `figure_caption` regions on each sheet
2. For each figure box, EasyOCR tries to read the label via a cascade of passes:
   top-left corner → top-right corner → below strip (350 px) → side margins
3. Each crop is saved as `_F<label>.png` (matched) or `_Fu.png` (needs review)
4. FAT files are always copied whole as `_Fu`

**Output naming:**

| Label source | Output filename | Meaning |
|---|---|---|
| OCR label match | `US…_F1A.png` | matched to FIG. 1A |
| Unmatched / FAT / error | `US…_Fu.png` | needs review |

Output is written to `matched/<patent_id>/` — raw files are never modified.

**Where it fits in the pipeline:**
```
00a  (PatSeer download)
 ↓
00b1 (grouping + batch assignment → batches.xlsx)
 ↓
00b2 ←  YOU ARE HERE  (run once per batch: descriptions CSV + DocLayout-YOLO → _F / _Fu labels)
 ↓
01   (human review for _Fu files)
 ↓
02   (pad + resize to 518×518)
```

---

| Cell | What it does |
|------|------|
| 1 | CUDA check |
| 2 | Load config + Excel, save descriptions.csv |
| 2b | Batch selector — set BATCH_INDEX and run |
| 3 | Build DocLayout-YOLO + EasyOCR engine (once per session) |
| 4 | Run pipeline over selected batch |
| 5 | Save review CSV + show flagged crops |

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("Is CUDA active inside Jupyter?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Connected GPU Model:", torch.cuda.get_device_name(0))

Is CUDA active inside Jupyter?: True
Connected GPU Model: NVIDIA GeForce RTX 2080 Ti


In [3]:
import sys
import os
import pandas as pd
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config_loader import load_config
from extractor import load_patseer_excel, save_descriptions_csv
import doclayout_matcher as dm

cfg = load_config()

print("Config loaded. Key paths:")
print("  raw_images :", cfg["paths"]["raw_images"])
print("  matched    :", cfg["paths"]["matched"])
print("  data       :", cfg["paths"]["data"])

Config loaded. Key paths:
  raw_images : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw
  matched    : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/matched
  data       : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data


In [4]:
excel_path = Path(cfg["paths"]["patseer_excel"])
print(f"Reading PatSeer data from: {excel_path.name}...")
patseer_index = load_patseer_excel(excel_path)

desc_csv_path = Path(cfg["paths"]["data"]) / "descriptions.csv"
save_descriptions_csv(patseer_index, desc_csv_path)

df = pd.read_csv(desc_csv_path)
print(f"descriptions.csv saved — {len(df)} records.")

Reading PatSeer data from: 1639__dataset_08_06_26.xlsx...


/home/vasco/anaconda3/envs/nb_00b2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1639__dataset_08_06_26.xlsx  (1639 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

In [5]:
# ── Cell 2b — Batch selector ──────────────────────────────────────────────────
# Loads batch assignments from data/batches.xlsx (produced by 00b1_grouping).
# Set BATCH_INDEX to run one batch at a time (0-based: 0, 1, 2, ...).
# Set BATCH_INDEX = None to run all patents.
#
# Run 00b1_grouping first — it creates batches.xlsx.

import pandas as pd
from pathlib import Path

BATCH_INDEX = 0     # ← change this to run different batches

batches_xlsx = Path(cfg["paths"]["data"]) / "batches.xlsx"
if not batches_xlsx.exists():
    raise FileNotFoundError(
        f"batches.xlsx not found at {batches_xlsx}\n"
        "Run 00b1_grouping.ipynb first to generate batch assignments."
    )

xl = pd.ExcelFile(batches_xlsx)
batch_sheet_names = [s for s in xl.sheet_names if s.startswith("Batch_")]
all_batches = [xl.parse(s, dtype=str) for s in batch_sheet_names]

print(f"Total patents   : {len(df)}")
print(f"Batches found   : {len(all_batches)}  (from {batches_xlsx.name})")
for i, b in enumerate(all_batches):
    print(f"  Batch {i}: {len(b)} patents")

if BATCH_INDEX is None:
    run_df = df
    print(f"\nRunning ALL {len(run_df)} patents.")
else:
    if BATCH_INDEX >= len(all_batches):
        raise ValueError(f"BATCH_INDEX {BATCH_INDEX} out of range — only {len(all_batches)} batches exist.")
    batch_df = all_batches[BATCH_INDEX]
    # batches.xlsx uses 'patent_id' column; filter df to only those patent_ids
    batch_ids = set(batch_df["patent_id"].dropna().str.strip())
    run_df = df[df["patent_id"].isin(batch_ids)].reset_index(drop=True)
    print(f"\nRunning Batch {BATCH_INDEX}: {len(run_df)} patents.")

Total patents   : 1639
Batches found   : 5  (from batches.xlsx)
  Batch 0: 352 patents
  Batch 1: 400 patents
  Batch 2: 305 patents
  Batch 3: 371 patents
  Batch 4: 211 patents

Running Batch 0: 352 patents.


In [6]:
# ── Cell 2c ── Limit run to N patents (mirrors LIMIT in 01_review.ipynb) ─────
# Set LIMIT to cap how many patents from the selected batch actually run
# through detection/cropping below. None runs every patent in the batch.
LIMIT = None   # int or None

if LIMIT:
    run_df = run_df.head(LIMIT)
    print(f"LIMIT={LIMIT} — running on the first {len(run_df)} patent(s) of this batch.")
else:
    print(f"LIMIT=None — running on all {len(run_df)} patent(s) of this batch.")


LIMIT=None — running on all 352 patent(s) of this batch.


In [7]:
# Engines are now built inside Cell 4 (dual-GPU). This cell is kept as a
# quick CUDA sanity check — run it to confirm both GPUs are visible.
import torch
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i}  {torch.cuda.get_device_name(i)}  "
          f"{torch.cuda.get_device_properties(i).total_memory // 1024**2} MB")


cuda:0  NVIDIA GeForce RTX 2080 Ti  10818 MB
cuda:1  NVIDIA GeForce RTX 2080 Ti  10821 MB


In [8]:
# ── GPU selection + VRAM check ─────────────────────────────────────────────────
# Run this before the cell below if you might be using a GPU for something else
# (another notebook, a training run, etc). It reports free VRAM per card and
# lets you pick which GPU(s) the dual-worker pipeline is allowed to use.
#
# GPU_SELECTION:
#   "auto"    -> use every GPU with enough free VRAM (>= MIN_FREE_GB), one worker each
#   "0"       -> force a single worker on GPU 0 only
#   "1"       -> force a single worker on GPU 1 only
#   "0,1"     -> force both GPUs regardless of current free memory (old default behaviour)
GPU_SELECTION = "0,1"  # "auto" or comma-separated list of GPU indices (0-based)
MIN_FREE_GB   = 6.0   # YOLO + EasyOCR + 4-bit Qwen2.5-VL-7B need roughly this much headroom

import torch

def _free_gb(idx: int) -> float:
    free_bytes, _total_bytes = torch.cuda.mem_get_info(idx)
    return free_bytes / 1e9

n_gpu = torch.cuda.device_count()
print(f"GPUs visible: {n_gpu}")
free_per_gpu = {}
for i in range(n_gpu):
    name = torch.cuda.get_device_name(i)
    free = _free_gb(i)
    free_per_gpu[i] = free
    flag = "OK" if free >= MIN_FREE_GB else "LOW"
    print(f"  GPU {i} ({name}): {free:.1f} GB free  [{flag}]")

if GPU_SELECTION == "auto":
    selected_gpu_ids = [str(i) for i in range(n_gpu) if free_per_gpu.get(i, 0) >= MIN_FREE_GB]
    if not selected_gpu_ids:
        best = max(free_per_gpu, key=free_per_gpu.get, default=0)
        selected_gpu_ids = [str(best)]
        print(f"\nNo GPU has >= {MIN_FREE_GB} GB free -- falling back to GPU {best} only "
              f"({free_per_gpu.get(best, 0):.1f} GB free). Expect possible OOM/'unavailable' "
              f"qwen_status rows; consider closing other GPU processes first.")
    else:
        print(f"\nAuto-selected GPU(s): {', '.join(selected_gpu_ids)}")
else:
    selected_gpu_ids = [s.strip() for s in GPU_SELECTION.split(",") if s.strip()]
    print(f"\nForced GPU selection: {', '.join(selected_gpu_ids)}")
    for s in selected_gpu_ids:
        idx = int(s)
        free = free_per_gpu.get(idx, 0)
        if free < MIN_FREE_GB:
            print(f"  WARNING: GPU {idx} only has {free:.1f} GB free (< {MIN_FREE_GB} GB) -- "
                  f"this worker may hit OOM. Close other processes on this GPU if possible.")

print(f"\nWorkers to launch: {len(selected_gpu_ids)} -> GPU(s) {selected_gpu_ids}")


GPUs visible: 2
  GPU 0 (NVIDIA GeForce RTX 2080 Ti): 10.3 GB free  [OK]
  GPU 1 (NVIDIA GeForce RTX 2080 Ti): 11.0 GB free  [OK]

Forced GPU selection: 0, 1

Workers to launch: 2 -> GPU(s) ['0', '1']


In [9]:
# -- GPU stale-process diagnostic (run after the GPU-selection cell above) --
# Memory held by a CUDA context is only released when its OWNING PROCESS
# exits -- torch.cuda.empty_cache() in THIS kernel cannot free memory that a
# different process (another notebook, another conda env, a crashed worker)
# is still holding. This cell just reports what nvidia-smi sees so you can
# tell "my own kernel is using this" from "something else left memory stuck"
# before you decide what to kill -- it does NOT kill anything automatically.
import subprocess, os

my_pid = os.getpid()

def _run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()

proc_csv = _run([
    "nvidia-smi", "--query-compute-apps=pid,used_memory,gpu_uuid",
    "--format=csv,noheader,nounits",
])
gpu_csv = _run(["nvidia-smi", "--query-gpu=index,uuid", "--format=csv,noheader"])
uuid_to_idx = {}
for line in gpu_csv.splitlines():
    idx, uuid = [p.strip() for p in line.split(",")]
    uuid_to_idx[uuid] = idx

print(f"This kernel's PID: {my_pid}\n")
print("GPU compute processes:")
stale = []
for line in proc_csv.splitlines():
    if not line.strip():
        continue
    pid_s, mem_s, uuid = [p.strip() for p in line.split(",")]
    pid = int(pid_s)
    gpu_idx = uuid_to_idx.get(uuid, "?")
    is_me = pid == my_pid
    status = _run(["ps", "-o", "stat=", "-p", str(pid)]) or "gone"
    cmd = _run(["ps", "-o", "cmd=", "-p", str(pid)])
    flag = "<- this kernel" if is_me else ("ZOMBIE/leaked" if status.startswith("Z") or not cmd else "")
    print(f"  GPU {gpu_idx}  PID {pid:>7}  {mem_s:>6} MiB  [{status or 'gone':<4}]  {flag}  {cmd[:70]}")
    if flag == "ZOMBIE/leaked":
        stale.append(pid)

if stale:
    print(f"\n{len(stale)} PID(s) look like leaked/zombie GPU memory from a "
          f"previous run: {stale}")
    print("These won't free themselves -- find their PARENT process "
          "(`ps -o ppid= -p <pid>`) and end IT (e.g. restart that other "
          "Jupyter kernel) once you've confirmed it's not work you still need.")
else:
    print("\nNo obviously leaked GPU processes found.")

This kernel's PID: 3710495

GPU compute processes:
  GPU 0  PID 3710495     154 MiB  [Sl  ]  <- this kernel  /home/vasco/anaconda3/envs/nb_00b2/bin/python -m ipykernel_launcher --
  GPU 1  PID    4026     156 MiB  [SLsl]    /usr/libexec/gnome-remote-desktop-daemon
  GPU 1  PID 3710495     154 MiB  [Sl  ]  <- this kernel  /home/vasco/anaconda3/envs/nb_00b2/bin/python -m ipykernel_launcher --

No obviously leaked GPU processes found.


In [10]:
import re, time, shutil, json

raw_dir       = Path(cfg["paths"]["raw_images"])
triage_dir    = Path(cfg["paths"]["triage"])
# Nest outputs under matched/<sheet_name>/ so different batches never collide or
# overwrite each other. Use the actual batches.xlsx SHEET NAME (e.g. "Batch_00"),
# not the raw BATCH_INDEX position — batch_sheet_names can contain extra sheets
# like "Batch_debug" before the numbered ones, so position != batch number.
# This also keeps the folder name identical to what 01_review.ipynb computes from
# its own BATCH_ID (f"Batch_{BATCH_ID:02d}"), so the two notebooks always agree.
batch_label   = batch_sheet_names[BATCH_INDEX] if BATCH_INDEX is not None else "all"
matched_dir   = Path(cfg["paths"]["matched"]) / batch_label
# data/ CSVs (crops_mapping.csv, needs_human_review.csv) are also nested per
# batch now — they used to live flat in data/ and get overwritten by the next
# batch run; nesting them removes that footgun entirely.
data_dir      = Path(cfg["paths"]["data_matched"]) / batch_label

# A run that gets interrupted (kernel stop, crash, Ctrl-C) leaves whatever
# patents it already finished sitting in matched_dir/data_dir. The next run
# then mixes old (possibly pre-fix or half-written) results with new ones,
# with no way to tell which is which. Wipe this batch's two output folders
# before every run so matched_dir/data_dir always reflect ONLY the run that
# just happened — set CLEAR_BEFORE_RUN = False if you ever want to resume on
# top of a previous run's output instead.
CLEAR_BEFORE_RUN = True
if CLEAR_BEFORE_RUN:
    if matched_dir.exists():
        shutil.rmtree(matched_dir)
    if data_dir.exists():
        shutil.rmtree(data_dir)
    print(f"CLEAR_BEFORE_RUN=True — wiped previous outputs for {batch_label}")

matched_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
print(f"Output folder for this run: {matched_dir}")
print(f"Data folder for this run  : {data_dir}")

scan_limit = cfg["extractor"].get("scan_limit", None)
if scan_limit:
    run_df = run_df.head(scan_limit)
print(f"Running on {len(run_df)} patents (scan_limit={scan_limit})...")

# ── Sheet/triage helpers (unchanged) ─────────────────────────────────────────
_CLEAN_RE   = re.compile(r"[^A-Za-z0-9]")
_NUM_SUFFIX = re.compile(r"_\d+$")
_DL_SUFFIX  = re.compile(r"PAFP$|PAF$", re.IGNORECASE)
_KIND_CODES = ["A1","A2","A3","B1","B2","C1","U1"]

def _core(pid):
    p = _NUM_SUFFIX.sub("", pid)
    c = _CLEAN_RE.sub("", p).upper()
    c = _DL_SUFFIX.sub("", c)
    for sfx in _KIND_CODES:
        if c.endswith(sfx):
            return c[:-len(sfx)]
    return c

folder_map = {_core(p.name): p for p in raw_dir.iterdir() if p.is_dir()}

_triage_excluded_cache: dict = {}
def _triage_excluded_files(patent_id):
    if patent_id in _triage_excluded_cache:
        return _triage_excluded_cache[patent_id]
    json_path = triage_dir / f"{patent_id}.json"
    excluded = set()
    if json_path.exists():
        try:
            data = json.loads(json_path.read_text())
            excluded = {fig["file"] for fig in data.get("figures", [])
                        if fig.get("keep") is False and fig.get("locked") is True}
        except Exception as e:
            print(f"  ⚠  Could not read triage JSON for {patent_id}: {e}")
    _triage_excluded_cache[patent_id] = excluded
    return excluded

_NON_SHEET = re.compile(r"manifest|thumbnail|cover|abstract|front.?page", re.IGNORECASE)
_SHEET_RE  = re.compile(r"""
    (?:
        _[Dd]\d{3,}          |   # _D00001
        PAFP_img\d           |   # US...PAFP_img1
        PAF_img\d            |   # WO...PAF_img1
        _img[af]?\d          |   # _img1 _imgf1 _imgaf1
        fig_\d               |   # fig_01
        record__fig_\d       |   # record__fig_01
        ^img[af]?\d          |   # imgf0001 imgaf001
        ^pat\d               |   # pat1.png
        ^FT_\d               |   # FT_1.png
        ^HDA\d               |   # HDA1.png
        ^\d+\.               |   # 1.png 2.png
        ^srep\d              |   # srep1.png
        sN_img\d                 # SDN...SN_img1
    )
""", re.VERBOSE | re.IGNORECASE)

def is_sheet(f):
    if f.suffix.lower() != ".png": return False
    if _NON_SHEET.search(f.name): return False
    return bool(_SHEET_RE.search(f.name))

# ── Run parallel processing (one subprocess per selected GPU) ─────────────────
import torch
print(f"Launching {len(selected_gpu_ids)} worker(s) on GPU(s) {selected_gpu_ids} — each builds its own engine")
t0 = time.time()

rows, triage_skipped_total = dm.process_patents_parallel(
    patent_rows            = run_df["patent_id"],
    folder_map             = folder_map,
    matched_dir            = matched_dir,
    triage_dir             = triage_dir,
    engines                = None,
    is_sheet_fn            = None,
    triage_excluded_fn     = None,
    cfg                    = cfg,
    gpu_ids                = selected_gpu_ids,  # from the GPU-selection cell above
)

elapsed = time.time() - t0
results_df = pd.DataFrame(rows)

crops_csv = data_dir / f"crops_mapping_{batch_label}.csv"
results_df.to_csv(crops_csv, index=False)

print(f"\nDone in {elapsed:.0f}s  ({elapsed/max(1,len(run_df)):.1f}s/patent)")
print(f"Total crops : {len(results_df)}")
if triage_skipped_total:
    print(f"Skipped (triage-rejected) sheets: {triage_skipped_total}")
if not results_df.empty:
    labelled_n = (results_df["needs_review"] == False).sum()
    print(f"Auto-labelled: {labelled_n}/{len(results_df)} = {100*labelled_n/len(results_df):.0f}%")
    print(results_df.groupby("method")["output"].count())
    if "review_hint" in results_df.columns:
        hint_n = (results_df["review_hint"] == "possible_multi_fig").sum()
        if hint_n:
            print(f"\nFlagged as possible multi-figure merges: {hint_n}")

CLEAR_BEFORE_RUN=True — wiped previous outputs for Batch_01
Output folder for this run: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/matched/Batch_01
Data folder for this run  : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/matched/Batch_01
Running on 352 patents (scan_limit=None)...
Launching 2 worker(s) on GPU(s) ['0', '1'] — each builds its own engine
[GPU 0] worker started (PID 3710648)
[GPU 1] worker started (PID 3710649)
[GPU 1] /home/vasco/anaconda3/envs/nb_00b2/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
[GPU 1]   warnings.warn(
[GPU 1] Loading local GPU Vision Model (4-bit): Qwen/Qwen2.5-VL-7B-Instruct...
[GPU 1] Using model cache repository path: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/models/Qwen
[GPU 0] Loading local GPU Vision Model (4-bit): Qwen/Qw

In [11]:
# ── Cell 4c — Match each crop to its description line ──────────────────────
# Moved here from 01_review.ipynb: that notebook used to re-derive OCR labels
# from filenames and call match_images() itself on EVERY Stage 01 run, even
# though everything match_images() needs — the filenames just produced above,
# and this patent's Brief Description text — is already sitting right here.
# Persists matched_description/match_status/etc. into crops_csv so
# 01_review.ipynb can just read them instead of recomputing them from scratch.
#
# match_images() only ever touches image_files[i].name (never opens the file),
# so plain Path(filename) stand-ins are enough — no need to resolve the real
# per-patent matched/ subfolder here.
from matcher import parse_description, match_images, label_from_filename
from sentence_transformers import SentenceTransformer

sbert_match = SentenceTransformer(
    "AI-Growth-Lab/PatentSBERTa", cache_folder=str(cfg["paths"]["sbert_cache"])
)
print("PatentSBERTa ready (figure-to-description-line matching).")

desc_df     = pd.read_csv(Path(cfg["paths"]["data"]) / "descriptions.csv", dtype=str).fillna("")
desc_lookup = dict(zip(desc_df["patent_id"], desc_df["description_of_drawings"]))

for col in ["matched_description", "match_status", "match_method",
            "match_confidence", "semantic_best_score", "fig_number", "duplicate_group"]:
    if col not in results_df.columns:
        results_df[col] = None

for pid in run_df["patent_id"]:
    mask = results_df["patent_id"] == pid
    rows = results_df.loc[mask]
    if rows.empty:
        continue

    image_files = [Path(fn) for fn in rows["output"]]
    ocr_labels  = [label_from_filename(fn) for fn in rows["output"]]
    parsed_desc = parse_description(desc_lookup.get(pid, ""), cfg)
    has_splits  = any(label is None for label in ocr_labels)

    match_results = match_images(
        image_files, ocr_labels, parsed_desc, cfg,
        sbert_model=sbert_match,
        total_desc_entries=len(parsed_desc),
        has_splits=has_splits,
    )

    for idx, res in zip(rows.index, match_results):
        results_df.loc[idx, "matched_description"] = res.get("matched_description")
        results_df.loc[idx, "match_status"]         = res.get("match_status")
        results_df.loc[idx, "match_method"]          = res.get("match_method")
        results_df.loc[idx, "match_confidence"]      = res.get("match_confidence")
        results_df.loc[idx, "semantic_best_score"]   = res.get("semantic_best_score")
        results_df.loc[idx, "fig_number"]            = res.get("fig_number")
        if "duplicate_group" in res:
            results_df.loc[idx, "duplicate_group"] = res["duplicate_group"]

results_df.to_csv(crops_csv, index=False)
print(f"Matched {results_df['match_status'].notna().sum()}/{len(results_df)} crops to description lines.")
if not results_df.empty:
    print(results_df["match_status"].value_counts())


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

PatentSBERTa ready (figure-to-description-line matching).
Matched 3546/3546 crops to description lines.
match_status
matched           1246
semantic          1025
unmatched          671
human_required     588
duplicate           16
Name: count, dtype: int64


In [12]:
if results_df.empty or "needs_review" not in results_df.columns:
    print("No crops produced — nothing to review.")
else:
    flagged_df = results_df[results_df["needs_review"] == True]
    review_csv = data_dir / f"needs_human_review_{batch_label}.csv"
    flagged_df.to_csv(review_csv, index=False)
    print(f"Saved {len(flagged_df)} flagged crops → {review_csv.name}")
    print(f"(Pass these to Step 01 — human review notebook)")
    if not flagged_df.empty:
        display(flagged_df[["patent_id","original","output","label","method"]].head(20))

Saved 588 flagged crops → needs_human_review_Batch_01.csv
(Pass these to Step 01 — human review notebook)


,patent_id,original,output,label,method
1,US2022267016A1,US2022267016A1_US20220267016A1_D00002.png,US2022267016A1_US20220267016A1_D00002_crop_0_F...,2,doclayout_easyocr
4,US11787551B1,US11787551PAFP_img1.png,US11787551PAFP_img1_crop_1_Fu.png,None,doclayout_easyocr
8,US11787551B1,US11787551PAFP_img2.png,US11787551PAFP_img2_crop_1_Fu.png,None,doclayout_easyocr
10,US11787551B1,US11787551PAFP_img4.png,US11787551PAFP_img4_crop_0_Fu.png,4,doclayout_easyocr
11,US11787551B1,US11787551PAFP_img4.png,US11787551PAFP_img4_crop_1_Fu.png,None,doclayout_easyocr
14,US11787551B1,US11787551PAFP_img8.png,US11787551PAFP_img8_crop_0_Fu.png,8,doclayout_qwen_strip
49,US2019031334A1,US20190031334PAFP_img5.png,US20190031334PAFP_img5_crop_1_Fu.png,2B,doclayout_easyocr
55,CA3096221A1,CA3096221_pct00001.png,CA3096221_pct00001_crop_0_Fu.png,None,doclayout_easyocr
56,CA3096221A1,CA3096221_pct00002.png,CA3096221_pct00002_crop_0_Fu.png,None,doclayout_easyocr
57,CA3096221A1,CA3096221_pct00003.png,CA3096221_pct00003_crop_0_Fu.png,None,doclayout_easyocr


In [13]:
# ── Cell 5b — Compute crop_quality flags from already-saved crops ───────────
# Pure post-processing: inspects PNGs already written to matched/, never
# touches raw/, never re-runs YOLO/EasyOCR. Appends crop_quality to
# crops_mapping.csv and overwrites it in place.
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import sys
from pathlib import Path as _Path
_notebook_dir  = _Path(__import__("os").getcwd())
_project_root  = _notebook_dir.parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from reviewer import resolve_patent_image_dir

crops_csv   = data_dir / f"crops_mapping_{batch_label}.csv"
# matched_dir is inherited from the batch-run cell above (already nested under
# matched/batch_<N>/ for this batch) — do not redefine it from the flat config path.
quality_df  = pd.read_csv(crops_csv)

BLANK_BRIGHTNESS_THRESH = 248
MIN_CROP_DIM_PX         = 80
LOW_CONF_THRESH         = 0.40

has_conf = "conf" in quality_df.columns
has_hint = "review_hint" in quality_df.columns

# matched/ folders are named "{patent_id}_{record_number}", not the bare
# patent_id — resolve via the same helper reviewer.py/run_stage01() use, and
# cache per patent_id since this is called once per row.
_patent_dir_cache: dict = {}

def _patent_dir(patent_id: str) -> Path:
    if patent_id not in _patent_dir_cache:
        _patent_dir_cache[patent_id] = resolve_patent_image_dir(matched_dir, str(patent_id))
    return _patent_dir_cache[patent_id]


def _crop_quality(row) -> str:
    img_path = _patent_dir(row["patent_id"]) / str(row["output"])
    if not img_path.exists():
        return "missing"
    try:
        with Image.open(img_path) as im:
            gray = np.asarray(im.convert("L"))
    except Exception:
        return "missing"

    h, w = gray.shape[:2]
    if float(gray.mean()) > BLANK_BRIGHTNESS_THRESH:
        return "blank"
    if min(h, w) < MIN_CROP_DIM_PX:
        return "too_small"
    if has_conf and pd.notna(row.get("conf")) and float(row["conf"]) < LOW_CONF_THRESH:
        return "low_conf"
    if has_hint and row.get("review_hint") == "possible_multi_fig":
        return "merged"
    return ""


quality_df["crop_quality"] = quality_df.apply(_crop_quality, axis=1)
quality_df.to_csv(crops_csv, index=False)

summary = quality_df["crop_quality"].fillna("").value_counts()
print(f"crop_quality summary ({len(quality_df)} crops):\n")
print(f"  {'crop_quality':<14}| count")
print(f"  {'-'*14}|------")
for label in ["", "blank", "too_small", "low_conf", "merged", "missing"]:
    display_label = "(empty)" if label == "" else label
    print(f"  {display_label:<14}| {int(summary.get(label, 0))}")

crop_quality summary (3546 crops):

  crop_quality  | count
  --------------|------
  (empty)       | 2486
  blank         | 670
  too_small     | 0
  low_conf      | 0
  merged        | 390
  missing       | 0


In [14]:
# ── Cell 5c — Interactive crop reviewer (Keep / Relabel / Reject) ───────────
# Shows only crops that need human attention (needs_review == True OR
# crop_quality != ""). Every action is written to crops_mapping.csv
# immediately, so progress survives kernel restarts. Files are renamed in
# matched/ only on Relabel — raw/ is never touched, YOLO/EasyOCR never re-run.
#
# Archived/opt-in: set RUN_REVIEWER = True to actually launch the UI. Left
# False by default so "Run All" doesn't pop up an interactive widget and block
# a hands-off batch run -- flip it on when you actually want to sit down and
# review a batch.
RUN_REVIEWER = False

import re
import base64
import pandas as pd
from pathlib import Path
from functools import partial
import ipywidgets as widgets
from IPython.display import display, clear_output
import sys
from pathlib import Path as _Path
_notebook_dir  = _Path(__import__("os").getcwd())
_project_root  = _notebook_dir.parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from reviewer import resolve_patent_image_dir

crops_csv   = data_dir / f"crops_mapping_{batch_label}.csv"
# matched_dir is inherited from the batch-run cell above (per-batch path) — do not
# redefine it from the flat config path.

results_df = pd.read_csv(crops_csv)
if "crop_quality" not in results_df.columns:
    results_df["crop_quality"] = ""
results_df["crop_quality"] = results_df["crop_quality"].fillna("")

# matched/ folders are named "{patent_id}_{record_number}" — resolve via the
# same helper reviewer.py/run_stage01() use, cached per patent_id.
_patent_dir_cache: dict = {}

def _patent_dir(patent_id: str) -> Path:
    if patent_id not in _patent_dir_cache:
        _patent_dir_cache[patent_id] = resolve_patent_image_dir(matched_dir, str(patent_id))
    return _patent_dir_cache[patent_id]


_BADGE_COLOR = {
    "blank": "#cf222e", "too_small": "#cf222e", "missing": "#cf222e",
    "low_conf": "#9a6700", "merged": "#9a6700",
}

flagged_mask  = (results_df["needs_review"] == True) | (results_df["crop_quality"] != "")
flagged_idxs  = set(results_df.index[flagged_mask])
flagged_patents = results_df.loc[flagged_mask, "patent_id"].drop_duplicates().tolist()
resolved_idxs = set()

state  = {"patent_idx": 0}
output = widgets.Output()

prev_btn = widgets.Button(description="◄ Prev")
next_btn = widgets.Button(description="Next ►")


def _b64(path: Path) -> str:
    return base64.b64encode(path.read_bytes()).decode()


def _badge_for(row) -> tuple[str, str]:
    q = row["crop_quality"]
    if q in _BADGE_COLOR:
        return _BADGE_COLOR[q], q
    if bool(row["needs_review"]):
        return "#c9b400", "needs_review"
    return "#888", q or "ok"


def _do_action(idx, *, new_quality=None, new_output=None, new_label=None, needs_review=None):
    if new_quality is not None:
        results_df.loc[idx, "crop_quality"] = new_quality
    if new_output is not None:
        results_df.loc[idx, "output"] = new_output
    if new_label is not None:
        results_df.loc[idx, "label"] = new_label
    if needs_review is not None:
        results_df.loc[idx, "needs_review"] = needs_review
    resolved_idxs.add(idx)
    results_df.to_csv(crops_csv, index=False)
    render()


def on_keep(idx, _btn):
    _do_action(idx, new_quality="")


def on_reject(idx, _btn):
    _do_action(idx, new_quality="rejected")


def on_recrop(idx, _btn):
    _do_action(idx, new_quality="re_crop")


def on_toggle_relabel(box, _btn):
    box.layout.display = "flex" if box.layout.display == "none" else "none"


def on_apply_relabel(idx, text_input, _btn):
    new_label = text_input.value.strip()
    if not new_label:
        return
    row = results_df.loc[idx]
    old_output = str(row["output"])
    if old_output.endswith("_Fu.png"):
        new_output = old_output[: -len("_Fu.png")] + f"_F{new_label}.png"
    else:
        new_output = re.sub(r"\.png$", f"_F{new_label}.png", old_output)

    patent_dir = _patent_dir(row["patent_id"])
    old_path = patent_dir / old_output
    new_path = patent_dir / new_output
    if old_path.exists():
        old_path.rename(new_path)

    _do_action(idx, new_quality="", new_output=new_output, new_label=new_label, needs_review=False)


def build_card(idx, row) -> widgets.VBox:
    img_path = _patent_dir(row["patent_id"]) / str(row["output"])
    badge_color, badge_text = _badge_for(row)
    label_text = row["label"] if pd.notna(row["label"]) else "Fu"
    img_b64 = _b64(img_path) if img_path.exists() else ""

    card_html = widgets.HTML(
        f"<div style='border:2px solid {badge_color}; border-radius:6px; padding:6px; "
        f"text-align:center;'>"
        f"<img src='data:image/png;base64,{img_b64}' "
        f"style='max-width:220px; max-height:200px; object-fit:contain;'/><br>"
        f"<span style='background:{badge_color}; color:white; padding:1px 6px; "
        f"border-radius:4px; font-size:11px;'>{badge_text}</span><br>"
        f"<b>{label_text}</b><br>"
        f"<span style='color:grey; font-size:10px;'>{row['crop_quality'] or 'needs_review'}</span>"
        f"</div>"
    )

    keep_btn    = widgets.Button(description="✓ Keep", button_style="success", layout=widgets.Layout(width="70px"))
    relabel_btn = widgets.Button(description="✎ Relabel", layout=widgets.Layout(width="80px"))
    reject_btn  = widgets.Button(description="✗ Reject", button_style="danger", layout=widgets.Layout(width="75px"))
    recrop_btn  = widgets.Button(description="✂ Re-crop", button_style="warning", layout=widgets.Layout(width="80px"))

    text_input = widgets.Text(placeholder="new label e.g. 3B", layout=widgets.Layout(width="120px"))
    apply_btn  = widgets.Button(description="Apply", button_style="info", layout=widgets.Layout(width="60px"))
    relabel_box = widgets.HBox([text_input, apply_btn], layout=widgets.Layout(display="none"))

    keep_btn.on_click(partial(on_keep, idx))
    reject_btn.on_click(partial(on_reject, idx))
    recrop_btn.on_click(partial(on_recrop, idx))
    relabel_btn.on_click(partial(on_toggle_relabel, relabel_box))
    apply_btn.on_click(partial(on_apply_relabel, idx, text_input))

    buttons_row = widgets.HBox([keep_btn, relabel_btn, reject_btn, recrop_btn])
    return widgets.VBox([card_html, buttons_row, relabel_box])


def change_patent(delta, _btn=None):
    state["patent_idx"] = max(0, min(state["patent_idx"] + delta, len(flagged_patents) - 1))
    render()


prev_btn.on_click(partial(change_patent, -1))
next_btn.on_click(partial(change_patent, 1))


def render():
    with output:
        clear_output(wait=True)
        if not flagged_idxs:
            display(widgets.HTML("<i>No crops need review — crops_mapping.csv is clean.</i>"))
            return

        pid = flagged_patents[state["patent_idx"]]
        recrop_count = int((results_df["crop_quality"] == "re_crop").sum())
        header = widgets.HTML(
            f"<h3>{len(flagged_idxs)} crops need attention ({len(flagged_patents)} patents)</h3>"
            f"<p>Reviewed: {len(resolved_idxs)} / {len(flagged_idxs)} &middot; Re-crop flagged: {recrop_count}</p>"
        )
        nav = widgets.HBox([
            prev_btn,
            widgets.Label(f"patent {state['patent_idx'] + 1}/{len(flagged_patents)} — {pid}"),
            next_btn,
        ])

        remaining = sorted(flagged_idxs - resolved_idxs)
        patent_rows = results_df.loc[[i for i in remaining if results_df.loc[i, "patent_id"] == pid]]

        if patent_rows.empty:
            grid = widgets.HTML("<i>All flagged crops for this patent are reviewed ✓</i>")
        else:
            cards = [build_card(idx, row) for idx, row in patent_rows.iterrows()]
            grid = widgets.GridBox(
                cards,
                layout=widgets.Layout(
                    grid_template_columns="repeat(3, 1fr)",
                    grid_gap="12px",
                    max_height="600px",
                    overflow_y="auto",
                ),
            )

        display(widgets.VBox([header, nav, grid]))


if RUN_REVIEWER:
    render()
    display(output)
else:
    print(f"RUN_REVIEWER is False — skipping interactive reviewer. "
          f"{len(flagged_idxs)} crop(s) across {len(flagged_patents)} patent(s) are flagged "
          f"and waiting for review when you're ready.")

RUN_REVIEWER is False — skipping interactive reviewer. 1474 crop(s) across 327 patent(s) are flagged and waiting for review when you're ready.


---
## Evaluation — inspect results inline

**Cell 6** — Summary stats (counts, labelling rate, method breakdown per patent)  
**Cell 7** — Visual grid: all crops for every patent, annotated with label + method  
**Cell 8** — Flagged-only reviewer: show just the `_Fu` crops that need human attention

In [15]:
# ── Cell 6 — Summary stats ────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

crops_csv = data_dir / f"crops_mapping_{batch_label}.csv"
results_df = pd.read_csv(crops_csv)

if results_df.empty:
    print("crops_mapping.csv is empty — run the pipeline first (Cell 4).")
else:
    total      = len(results_df)
    labelled_n = (results_df["needs_review"] == False).sum()
    flagged_n  = (results_df["needs_review"] == True).sum()

    print(f"{'Total crops':<25} {total}")
    print(f"{'Auto-labelled (_F)':<25} {labelled_n}  ({100*labelled_n/total:.0f}%)")
    print(f"{'Needs review (_Fu)':<25} {flagged_n}  ({100*flagged_n/total:.0f}%)")
    print()

    # Per-patent breakdown
    per_patent = (
        results_df
        .groupby("patent_id")
        .agg(
            sheets   = ("original", "nunique"),
            crops    = ("output", "count"),
            labelled = ("needs_review", lambda x: (x == False).sum()),
            flagged  = ("needs_review", lambda x: (x == True).sum()),
        )
        .assign(pct=lambda d: (d["labelled"] / d["crops"] * 100).round(0).astype(int))
        .rename(columns={"pct": "labelled_%"})
        .sort_values("labelled_%")
    )
    display(per_patent)

    print()
    print("Method breakdown:")
    display(results_df.groupby("method")["output"].count().rename("crops").to_frame())

Total crops               3546
Auto-labelled (_F)        2958  (83%)
Needs review (_Fu)        588  (17%)



,sheets,crops,labelled,flagged,labelled_%
patent_id,,,,,
KR20250121294A,6,6,0,6,0
CN120397241A,5,5,0,5,0
CN115303479A,6,6,0,6,0
CN113525678A,4,4,0,4,0
CN113086184A,2,2,0,2,0
...,...,...,...,...,...
US2019277353A1,4,6,6,0,100
US2019283858A1,6,6,6,0,100
US2019291861A1,7,7,7,0,100



Method breakdown:


,crops
method,
doclayout_easyocr,2985
doclayout_qwen,444
doclayout_qwen_strip,117


In [ ]:
# ── Cell 7 — Crops viewer (3-column HTML grid, ← → patent navigation) ────────
import re, base64, textwrap
import pandas as pd
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# matched_dir is inherited from the batch-run cell above (per-batch path).
crops_df    = pd.read_csv(data_dir / f"crops_mapping_{batch_label}.csv")
desc_df     = pd.read_csv(Path(cfg["paths"]["data"]) / "descriptions.csv")
desc_lookup = desc_df.set_index("patent_id")["description_of_drawings"].to_dict()

_FIG_RE = re.compile(
    r"FIG\.?\s*(\d+\s*[A-Za-z]?)\s+(?:and\s+FIG\.?\s*\d+\s*[A-Za-z]?\s+)?(?:is|are|show|illustrate|depict)\b",
    re.IGNORECASE,
)

def parse_fig_descriptions(text):
    if not isinstance(text, str): return {}
    text = re.sub(r"\s*\n\s*", " ", text)
    parts = re.split(r"(?=FIG\.?\s*\d)", text, flags=re.IGNORECASE)
    result = {}
    for part in parts:
        m = _FIG_RE.search(part)
        if m:
            key = re.sub(r"\s+", "", m.group(1)).upper()
            result.setdefault(key, re.sub(r"\s{2,}", " ", part).strip()[:300])
    return result

def _b64(path):
    return base64.b64encode(path.read_bytes()).decode()

patents = list(crops_df["patent_id"].unique())
if not patents:
    print("No crops yet — run the pipeline (Cell 4) first.")
else:
    state    = {"idx": 0}
    counter  = widgets.HTML()
    prev_btn = widgets.Button(description="◀ Prev", button_style="info",
                              layout=widgets.Layout(width="100px"))
    next_btn = widgets.Button(description="Next ▶", button_style="info",
                              layout=widgets.Layout(width="100px"))
    out      = widgets.Output()

    # Build a lookup: output filename → absolute path, searching all matched locations
    base_dir = Path(cfg["paths"]["base"])
    matched_root = Path(cfg["paths"]["matched"])
    all_matched_roots = (
        [matched_dir]
        + sorted(matched_root.glob("batch_*"))   # other batches under matched/batch_<N>/
        + sorted(base_dir.glob("*/matched"))      # legacy batch_00-style collected folders
    )
    _crop_file_index: dict = {}
    for root in all_matched_roots:
        if not root.exists(): continue
        for patent_folder in root.iterdir():
            if not patent_folder.is_dir(): continue
            for f in patent_folder.iterdir():
                _crop_file_index[f.name] = f

    def _find_crop(filename: str):
        return _crop_file_index.get(filename)

    def render(idx):
        patent_id = patents[idx]
        grp       = crops_df[crops_df["patent_id"] == patent_id]
        fig_descs = parse_fig_descriptions(desc_lookup.get(patent_id, ""))
        out_dir   = None  # unused now — we use _find_crop() per file

        n_ok = (grp["needs_review"] == False).sum()
        n_fu = (grp["needs_review"] == True).sum()

        cards = ""
        for _, row in grp.iterrows():
            img_path = _find_crop(row["output"])
            img_tag  = ""
            if img_path and img_path.exists():
                img_tag = f'<img src="data:image/png;base64,{_b64(img_path)}" style="max-width:100%;max-height:260px;object-fit:contain;"/>'

            label_key = str(row["label"]).upper() if pd.notna(row["label"]) else None
            hint      = row.get("review_hint") or ""

            if not row["needs_review"]:
                badge_bg, badge_col = "#d4edda", "#155724"
                badge_txt = f"&#10003; MATCHED &nbsp;&nbsp; FIG.&nbsp;{row['label']}"
                desc_html = ""
                if label_key and label_key in fig_descs:
                    desc_html = f'<p style="font-size:10px;color:#155724;margin:4px 0 0 0;">{fig_descs[label_key]}</p>'
            elif hint == "possible_multi_fig":
                badge_bg, badge_col = "#fff3cd", "#856404"
                badge_txt = "&#9888; POSSIBLE MULTI-FIG"
                desc_html = '<p style="font-size:10px;color:#856404;margin:4px 0 0 0;">May contain multiple figures — check and rename</p>'
            else:
                badge_bg, badge_col = "#f8d7da", "#721c24"
                badge_txt = "&#9888; UNMATCHED — rename _Fu &#8594; _F&lt;n&gt;"
                desc_html = '<p style="font-size:10px;color:#721c24;margin:4px 0 0 0;">No label found — needs human review</p>'

            cards += f"""
            <div style="border:1px solid #ccc;border-radius:6px;padding:6px;
                        display:flex;flex-direction:column;align-items:center;gap:4px;">
                <div style="background:{badge_bg};color:{badge_col};border:1px solid {badge_col};
                            border-radius:4px;padding:3px 8px;font-size:11px;font-weight:bold;
                            width:100%;text-align:center;">
                    {badge_txt}
                </div>
                <div style="font-size:9px;color:#666;text-align:center;word-break:break-all;">
                    {row["output"]}
                </div>
                {img_tag}
                {desc_html}
            </div>"""

        html = f"""
        <div>
            <h3 style="margin:4px 0;">{patent_id}
                <span style="font-size:12px;font-weight:normal;color:#555;">
                    &nbsp;—&nbsp;{len(grp)} crops &nbsp;|&nbsp;
                    <span style="color:#155724;">{n_ok} matched</span> &nbsp;|&nbsp;
                    <span style="color:#721c24;">{n_fu} unmatched</span>
                </span>
            </h3>
            <details style="margin-bottom:8px;">
                <summary style="cursor:pointer;color:#555;font-size:11px;">Description ({len(fig_descs)} FIG. entries)</summary>
                <p style="font-size:11px;color:#333;max-width:900px;">
                    {str(desc_lookup.get(patent_id,""))[:1200]}
                </p>
            </details>
            <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;">
                {cards}
            </div>
        </div>"""

        counter.value = (f"<span style='color:grey;font-size:12px;'>"
                         f"Patent {idx+1} / {len(patents)}</span>")
        prev_btn.disabled = (idx == 0)
        next_btn.disabled = (idx == len(patents) - 1)

        with out:
            clear_output(wait=True)
            display(HTML(html))

    def on_prev(_): state["idx"] = max(0, state["idx"]-1); render(state["idx"])
    def on_next(_): state["idx"] = min(len(patents)-1, state["idx"]+1); render(state["idx"])
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)

    nav = widgets.HBox([prev_btn, counter, next_btn],
                       layout=widgets.Layout(align_items="center", gap="12px"))
    display(widgets.VBox([nav, out]))
    render(0)


In [ ]:
# ── Cell 8 — All crops viewer (3-column HTML grid) ───────────────────────────
import re, base64, textwrap
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# matched_dir is inherited from the batch-run cell above (per-batch path).
crops_df    = pd.read_csv(data_dir / f"crops_mapping_{batch_label}.csv")
desc_df     = pd.read_csv(Path(cfg["paths"]["data"]) / "descriptions.csv")
desc_lookup = desc_df.set_index("patent_id")["description_of_drawings"].to_dict()

def _b64(path):
    return base64.b64encode(path.read_bytes()).decode()

def clean_desc(text, max_chars=1200):
    if not isinstance(text, str): return "(no description)"
    return re.sub(r"\s{2,}", " ", text).strip()[:max_chars]

for patent_id, grp in crops_df.groupby("patent_id"):
    folders = [p for p in matched_dir.iterdir() if p.name.startswith(patent_id)]
    if not folders: continue
    out_dir = folders[0]

    n_ok  = (grp["needs_review"] == False).sum()
    n_fu  = (grp["needs_review"] == True).sum()
    desc  = clean_desc(desc_lookup.get(patent_id, ""))

    cards = ""
    for _, row in grp.iterrows():
        img_path = out_dir / row["output"]
        if not img_path.exists():
            continue
        b64   = _b64(img_path)
        label = row.get("label") or ""
        hint  = row.get("review_hint") or ""  
        needs_review = row["needs_review"]

        if not needs_review:
            badge_bg  = "#d4edda"
            badge_col = "#155724"
            badge_txt = f"&#10003; MATCHED &nbsp; {label}"
        elif hint == "possible_multi_fig":
            badge_bg  = "#fff3cd"
            badge_col = "#856404"
            badge_txt = "&#9888; POSSIBLE MULTI-FIG"
        else:
            badge_bg  = "#f8d7da"
            badge_col = "#721c24"
            badge_txt = "&#9888; UNMATCHED — needs review"

        cards += f"""
        <div style="border:1px solid #ccc; border-radius:6px; padding:6px;
                    display:flex; flex-direction:column; align-items:center; gap:4px;">
            <div style="background:{badge_bg}; color:{badge_col};
                        border:1px solid {badge_col}; border-radius:4px;
                        padding:3px 8px; font-size:11px; font-weight:bold;
                        width:100%; text-align:center;">
                {badge_txt}
            </div>
            <div style="font-size:10px; color:#555; text-align:center; word-break:break-all;">
                {row["output"]}
            </div>
            <img src="data:image/png;base64,{b64}"
                 style="max-width:100%; max-height:280px; object-fit:contain;"/>
        </div>"""

    html = f"""
    <div style="margin-bottom:30px;">
        <h3 style="margin:0 0 4px 0;">{patent_id}
            <span style="font-size:13px; font-weight:normal; color:#555;">
                &nbsp;—&nbsp; {len(grp)} crops &nbsp;|&nbsp;
                <span style="color:#155724;">{n_ok} matched</span> &nbsp;|&nbsp;
                <span style="color:#721c24;">{n_fu} unmatched</span>
            </span>
        </h3>
        <details><summary style="cursor:pointer; color:#555; font-size:12px;">Description</summary>
            <p style="font-size:11px; color:#333; max-width:900px;">{desc}</p>
        </details>
        <div style="display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-top:10px;">
            {cards}
        </div>
    </div>
    <hr/>"""

    display(HTML(html))


## Cell 9 — Duplicate-label audit

Flags two suspicious patterns in `crops_mapping.csv` that signal labelling problems:

1. **Same label on multiple crops of the same sheet** — usually means the sheet has
   side-by-side sub-figures that the vertical splitter kept in one crop, and the OCR
   cascade read the same caption for each row (e.g. three crops all labelled `2A`).
2. **Low label diversity per patent** — many crops but few distinct labels suggests
   the cascade is picking up a neighbouring figure's caption.

These are review hints only — nothing is modified.

In [ ]:
# ── Cell 9 — Duplicate-label audit ────────────────────────────────────────────
import pandas as pd
from pathlib import Path

crops_df = pd.read_csv(data_dir / f"crops_mapping_{batch_label}.csv")
labelled = crops_df[crops_df["needs_review"] == False].copy()

if labelled.empty:
    print("No labelled crops yet — run the pipeline (Cell 4) first.")
else:
    # 1) Same label repeated on the same sheet
    dup_sheet = (
        labelled.groupby(["patent_id", "original", "label"])
        .size().rename("count").reset_index()
        .query("count > 1")
        .sort_values("count", ascending=False)
    )
    print(f"Same label on multiple crops of one sheet: {len(dup_sheet)} cases")
    if not dup_sheet.empty:
        display(dup_sheet.head(20))

    # 2) Label diversity per patent (crops vs distinct labels)
    diversity = (
        labelled.groupby("patent_id")
        .agg(crops=("output", "count"), distinct_labels=("label", "nunique"))
        .assign(ratio=lambda d: (d["distinct_labels"] / d["crops"]).round(2))
        .sort_values("ratio")
    )
    low = diversity[diversity["ratio"] < 0.7]
    print(f"\nPatents with low label diversity (<70% unique): {len(low)}")
    if not low.empty:
        display(low)

    # 3) Boxes flagged as possibly containing multiple merged sub-figures
    # (e.g. FIG.8A/8B/8C wrapped in one YOLO box — see review_hint in doclayout_matcher.py).
    # Flag only — crop/label is untouched; just helps reviewers spot these fast.
    if "review_hint" in crops_df.columns:
        multi_fig = crops_df[crops_df["review_hint"] == "possible_multi_fig"]
        print(f"\nFlagged as possible multi-figure merges: {len(multi_fig)}")
        if not multi_fig.empty:
            display(multi_fig[["patent_id", "original", "output", "label", "needs_review"]])

Same label on multiple crops of one sheet: 0 cases

Patents with low label diversity (<70% unique): 4


,crops,distinct_labels,ratio
patent_id,,,
US2023257114A1,9,1,0.11
CN108688803A,3,1,0.33
US2021188427A1,2,1,0.50
AU2020100605A4,11,7,0.64



Flagged as possible multi-figure merges: 338


,patent_id,original,output,label,needs_review
6,US2022234745A1,US20220234745A1_D00006.png,US20220234745A1_D00006_crop_0_F7.png,7,False
8,US2022234745A1,US20220234745A1_D00008.png,US20220234745A1_D00008_crop_0_Fu.png,NaN,True
11,US2021403155A1,US20210403155A1_D00001.png,US20210403155A1_D00001_crop_0_F1.png,1,False
14,US2021403155A1,US20210403155A1_D00004.png,US20210403155A1_D00004_crop_0_F4.png,4,False
15,US2021403155A1,US20210403155A1_D00005.png,US20210403155A1_D00005_crop_0_F5.png,5,False
...,...,...,...,...,...
2608,WO2015094020A2,WO2015094020PAFP_img1.png,WO2015094020PAFP_img1_crop_1_Fu.png,NaN,True
2609,WO2015094020A2,WO2015094020PAFP_img2.png,WO2015094020PAFP_img2_crop_0_Fu.png,NaN,True
2611,WO2015094020A2,WO2015094020PAFP_img2.png,WO2015094020PAFP_img2_crop_2_Fu.png,NaN,True
2613,US2025238031A1,US20250238031A1_D00003.png,US20250238031A1_D00003_crop_0_F3.png,3,False


Same label on multiple crops of one sheet: 0 cases

Patents with low label diversity (<70% unique): 1


,crops,distinct_labels,ratio
patent_id,,,
EP3770063A1,3,2,0.67



Flagged as possible multi-figure merges: 494


,patent_id,original,output,label,needs_review
3,US11787551B1,US11787551PAFP_img1.png,US11787551PAFP_img1_crop_0_F1.png,1,False
5,US11787551B1,US11787551PAFP_img10.png,US11787551PAFP_img10_crop_0_F10A.png,10A,False
6,US11787551B1,US11787551PAFP_img11.png,US11787551PAFP_img11_crop_0_F11.png,11,False
7,US11787551B1,US11787551PAFP_img2.png,US11787551PAFP_img2_crop_0_F2.png,2,False
18,US2015014475A1,US20150014475PAFP_img10.png,US20150014475PAFP_img10_crop_1_F8A.png,8A,False
...,...,...,...,...,...
3506,US2023067713A1,US20230067713A1_D00007.png,US20230067713A1_D00007_crop_1_F15.png,15,False
3507,US2023067713A1,US20230067713A1_D00007.png,US20230067713A1_D00007_crop_2_F14.png,14,False
3508,US2023067713A1,US20230067713A1_D00008.png,US20230067713A1_D00008_crop_0_Fu.png,NaN,True
3513,US2024294246A1,US20240294246A1_D00004.png,US20240294246A1_D00004_crop_0_F4.png,4,False


In [ ]:
# ── Collect batch outputs → batch_<N>/ ────────────────────────────────────────
# crops_mapping.csv / needs_human_review.csv are already isolated per batch
# (data/<Batch_NN>/, with the batch name baked into the filename too), and
# matched/ crops are already isolated per batch (matched/<Batch_NN>/) — so
# nothing here is at risk of being overwritten by the next batch run anymore.
# This cell is now just a convenience: one self-contained folder for sharing
# or archiving a batch's outputs.
#
# Disabled by default — matched/<Batch_NN>/ and data/matched/<Batch_NN>/ are
# already the live, canonical outputs that every other cell/notebook reads.
# This cell only ever produced a redundant full copy under base/<Batch_NN>/
# with nothing downstream consuming it — flip RUN_COLLECT_MIRROR to True only
# if you specifically want to bundle a batch for sharing/archiving.
RUN_COLLECT_MIRROR = False

if RUN_COLLECT_MIRROR:
    import shutil
    from pathlib import Path
    from reviewer import resolve_patent_image_dir

    BATCH_OUT = Path(cfg["paths"]["base"]) / batch_label

    # 1. matched crops for this batch
    matched_dst = BATCH_OUT / "matched"
    matched_dst.mkdir(parents=True, exist_ok=True)
    for pid in run_df["patent_id"]:
        # matched/ folders are named '{patent_id}_{record_number}', not the bare
        # patent_id — resolve via the shared helper (same fix as cells 5b/5c) so
        # this copy step actually finds the crops instead of silently no-opping.
        src = resolve_patent_image_dir(matched_dir, str(pid))
        if src.exists():
            dst = matched_dst / src.name
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)

    # 2. data CSVs for this batch (already at data_dir, just mirrored here)
    data_dst = BATCH_OUT / "data"
    data_dst.mkdir(parents=True, exist_ok=True)
    for csv_name in ["descriptions.csv", f"crops_mapping_{batch_label}.csv", f"needs_human_review_{batch_label}.csv"]:
        p = Path(cfg["paths"]["data"]) / csv_name if csv_name == "descriptions.csv" else data_dir / csv_name
        if p.exists():
            shutil.copy2(p, data_dst / csv_name)

    print(f"Batch outputs collected → {BATCH_OUT}")
    print(f"  matched/ : {len(list(matched_dst.iterdir()))} patent folders")
    print(f"  data/    : {[f.name for f in data_dst.iterdir()]}")
else:
    print("RUN_COLLECT_MIRROR is False — skipping redundant batch-output mirror copy.")

RUN_COLLECT_MIRROR is False — skipping redundant batch-output mirror copy.
